In [1]:
import os
import xarray as xr


# # Directory
# # Team members: Does not work if you don't add a shortcut for "Hackathon II Group 2" to your MyDrive from your "Shared with me"
# folder_path = '/content/drive/MyDrive/Hackathon II Group 2/'
# os.chdir(folder_path)

# Datasets
ds_flood_terrain_northumbria = xr.open_dataset(
    "flood_risk_terrain_northumbria.nc",
    engine="netcdf4"
)
ds_flood_terrain_severn = xr.open_dataset('flood_risk_terrain_severn.nc', engine="netcdf4")
ds_era5_severn = xr.open_dataset('era5_land_severn.nc', engine="netcdf4")
ds_era5_northumbria = xr.open_dataset('era5_land_northumbria.nc', engine="netcdf4")


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Convert to dataframes ───────────────────────────────────────────
df_severn = ds_flood_terrain_severn.to_dataframe().reset_index()
df_northumbria = ds_flood_terrain_northumbria.to_dataframe().reset_index()

print("Severn shape:", df_severn.shape)
print("Northumbria shape:", df_northumbria.shape)
print("\nColumns:", df_severn.columns.tolist())

# ── Filter to risk pixels ───────────────────────────────────────────
risk_cols = ['risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m']

def filter_risk_pixels(df):
    mask = df[risk_cols].notna().any(axis=1) & (df[risk_cols] != 0).any(axis=1)
    return df[mask].copy()

df_s = filter_risk_pixels(df_severn)
df_n = filter_risk_pixels(df_northumbria)

print(f"\nSevern   — total: {len(df_severn):,} | risk pixels: {len(df_s):,} ({len(df_s)/len(df_severn)*100:.1f}%)")
print(f"Northumbria — total: {len(df_northumbria):,} | risk pixels: {len(df_n):,} ({len(df_n)/len(df_northumbria)*100:.1f}%)")

# ── Class distributions ─────────────────────────────────────────────
print("\n── Severn risk_0_2m distribution ──")
print(df_s['risk_0_2m'].value_counts().sort_index())

print("\n── Northumbria risk_0_2m distribution ──")
print(df_n['risk_0_2m'].value_counts().sort_index())

# ── Feature engineering ─────────────────────────────────────────────
for df in [df_s, df_n]:
    df['log_flow_acc'] = np.log1p(df['flow_acc'])
    df['dtm_m'] = df['dtm'] / 10
    df['dtm_zscore'] = (df['dtm_m'] - df['dtm_m'].mean()) / df['dtm_m'].std()
    df['waw'] = df['waw'].where(df['waw'] <= 4, np.nan).fillna(0)
    df['imd'] = df['imd'].where(df['imd'] <= 100, np.nan).fillna(0)
    df['is_waterway'] = df['rciw'].notna().astype(int)

# ── Correlations with risk_0_2m ─────────────────────────────────────
features = ['dtm_m', 'dtm_zscore', 'log_flow_acc', 'imd', 'waw', 'is_waterway']

print("\n── Severn correlations with risk_0_2m ──")
print(df_s[features + ['risk_0_2m']].corr()['risk_0_2m'].sort_values())

print("\n── Northumbria correlations with risk_0_2m ──")
print(df_n[features + ['risk_0_2m']].corr()['risk_0_2m'].sort_values())

# ── ERA5 quick look ─────────────────────────────────────────────────
print("\n── ERA5 Severn structure ──")
print(ds_era5_severn)
print("\nERA5 Severn time range:")
#print(ds_era5_severn.time.values[[0, -1]])

Severn shape: (83959808, 15)
Northumbria shape: (46023329, 15)

Columns: ['y', 'x', 'dtm', 'flow_dir', 'flow_acc', 'imd', 'waw', 'rciw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'spatial_ref']

Severn   — total: 83,959,808 | risk pixels: 3,441,658 (4.1%)
Northumbria — total: 46,023,329 | risk pixels: 1,416,255 (3.1%)

── Severn risk_0_2m distribution ──
risk_0_2m
0.0     111943
1.0     847433
2.0     761336
3.0     435579
4.0    1285362
Name: count, dtype: int64

── Northumbria risk_0_2m distribution ──
risk_0_2m
1.0    249498
2.0    395010
3.0    207453
4.0    564289
Name: count, dtype: int64

── Severn correlations with risk_0_2m ──
dtm_m          -0.145721
dtm_zscore     -0.145721
imd            -0.115938
log_flow_acc    0.038803
is_waterway     0.055804
waw             0.087570
risk_0_2m       1.000000
Name: risk_0_2m, dtype: float64

── Northumbria correlations with risk_0_2m ──
dtm_m          -0.188043
dtm_zscore     -0.188043
imd            -0

In [4]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

# ── 1. Load datasets ─────────────────────────────────────────────────
ds_flood_terrain_northumbria = xr.open_dataset('flood_risk_terrain_northumbria.nc')
ds_flood_terrain_severn = xr.open_dataset('flood_risk_terrain_severn.nc')
ds_era5_severn = xr.open_dataset('era5_land_severn.nc')
ds_era5_northumbria = xr.open_dataset('era5_land_northumbria.nc')

# ── 2. Convert to dataframes ─────────────────────────────────────────
df_severn = ds_flood_terrain_severn.to_dataframe().reset_index()
df_northumbria = ds_flood_terrain_northumbria.to_dataframe().reset_index()

print("Severn shape:", df_severn.shape)
print("Northumbria shape:", df_northumbria.shape)

# ── 3. Filter to valid risk pixels ───────────────────────────────────
risk_cols = ['risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m']

def filter_risk_pixels(df):
    mask = df[risk_cols].notna().any(axis=1) & (df[risk_cols] != 0).any(axis=1)
    return df[mask].copy()

df_s = filter_risk_pixels(df_severn)
df_n = filter_risk_pixels(df_northumbria)

# Drop class 0 sentinel from Severn
df_s = df_s[df_s['risk_0_2m'].isin([1., 2., 3., 4.])].copy()

print(f"\nSevern risk pixels:      {len(df_s):,}")
print(f"Northumbria risk pixels: {len(df_n):,}")

# ── 4. Base feature engineering ──────────────────────────────────────
def engineer_features(df):
    df = df.copy()
    
    # Clean sentinel values
    df['waw'] = df['waw'].where(df['waw'] <= 4, np.nan).fillna(0)
    df['imd'] = df['imd'].where(df['imd'] <= 100, np.nan).fillna(0)
    
    # Terrain features
    df['dtm_m']       = df['dtm'] / 10
    df['dtm_zscore']  = (df['dtm_m'] - df['dtm_m'].mean()) / df['dtm_m'].std()
    df['log_flow_acc'] = np.log1p(df['flow_acc'])
    
    # Waterway indicator
    df['is_waterway'] = df['rciw'].notna().astype(int)
    
    # Land cover
    df['clc_type_clean'] = df['clc_type'].fillna(df['clc_type'].median())
    
    # Engineered features
    df['log_flow_acc_sq'] = df['log_flow_acc'] ** 2
    df['flow_acc_pct']    = df['log_flow_acc'] / df['log_flow_acc'].max()
    df['elev_x_flow']     = df['dtm_zscore'] * df['log_flow_acc']
    
    return df

df_s = engineer_features(df_s)
df_n = engineer_features(df_n)

print("\nFeature engineering done.")
print("Sample df_s columns:", df_s.columns.tolist())

# ── 5. Class distribution check ───────────────────────────────────────
print("\nSevern risk_0_2m distribution:")
print(df_s['risk_0_2m'].value_counts().sort_index())

print("\nNorthumbria risk_0_2m distribution:")
print(df_n['risk_0_2m'].value_counts().sort_index())

# ── 6. Class weights ──────────────────────────────────────────────────
classes = np.array([1., 2., 3., 4.])
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=df_s['risk_0_2m'].values
)
print("\nClass weights:")
for c, w in zip(classes, weights):
    print(f"  Class {int(c)}: {w:.3f}")

# ── 7. Model v1 — terrain only baseline ──────────────────────────────
features_v1 = ['dtm_zscore', 'log_flow_acc', 'imd', 'waw', 'is_waterway']

df_sample1 = df_s.dropna(subset=features_v1 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)
X1 = df_sample1[features_v1].values
y1 = (df_sample1['risk_0_2m'] - 1).astype(int).values

X_train1, X_val1, y_train1, y_val1 = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

model1 = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    n_jobs=-1, random_state=42, eval_metric='mlogloss', verbosity=0
)
model1.fit(X_train1, y_train1,
           eval_set=[(X_val1, y_val1)], verbose=False)

y_pred1 = model1.predict(X_val1)
print("\n── v1: Severn validation (seen) ──")
print(f"Accuracy: {accuracy_score(y_val1, y_pred1):.3f}")
print(classification_report(y_val1, y_pred1,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

df_n1 = df_n.dropna(subset=features_v1 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)
y_pred_n1 = model1.predict(df_n1[features_v1].values)
y_test1 = (df_n1['risk_0_2m'] - 1).astype(int).values
print("── v1: Northumbria test (unseen) ──")
print(f"Accuracy: {accuracy_score(y_test1, y_pred_n1):.3f}")
print(classification_report(y_test1, y_pred_n1,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

# ── 8. Model v2 — expanded terrain features ───────────────────────────
features_v2 = [
    'dtm_zscore', 'log_flow_acc', 'log_flow_acc_sq',
    'flow_acc_pct', 'elev_x_flow',
    'imd', 'waw', 'is_waterway', 'clc_type_clean'
]

df_sample2 = df_s.dropna(subset=features_v2 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)
X2 = df_sample2[features_v2].values
y2 = (df_sample2['risk_0_2m'] - 1).astype(int).values

X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

model2 = XGBClassifier(
    n_estimators=300, max_depth=7, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, eval_metric='mlogloss', verbosity=0
)
model2.fit(X_train2, y_train2,
           eval_set=[(X_val2, y_val2)], verbose=False)

y_pred2 = model2.predict(X_val2)
print("\n── v2: Severn validation (seen) ──")
print(f"Accuracy: {accuracy_score(y_val2, y_pred2):.3f}")
print(classification_report(y_val2, y_pred2,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

df_n2 = df_n.dropna(subset=features_v2 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)
X_test2 = df_n2[features_v2].values
y_test2 = (df_n2['risk_0_2m'] - 1).astype(int).values

y_pred_n2 = model2.predict(X_test2)
print("── v2: Northumbria test (unseen) ──")
print(f"Accuracy: {accuracy_score(y_test2, y_pred_n2):.3f}")
print(classification_report(y_test2, y_pred_n2,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

# ── 9. Feature importance v2 ──────────────────────────────────────────
feat_imp = pd.Series(model2.feature_importances_, index=features_v2)
print("\n── Feature importance v2 ──")
print(feat_imp.sort_values(ascending=False))

Severn shape: (83959808, 15)
Northumbria shape: (46023329, 15)

Severn risk pixels:      3,329,710
Northumbria risk pixels: 1,416,255

Feature engineering done.
Sample df_s columns: ['y', 'x', 'dtm', 'flow_dir', 'flow_acc', 'imd', 'waw', 'rciw', 'clc_type', 'risk_0_2m', 'risk_0_3m', 'risk_0_6m', 'risk_0_9m', 'risk_1_2m', 'spatial_ref', 'dtm_m', 'dtm_zscore', 'log_flow_acc', 'is_waterway', 'clc_type_clean', 'log_flow_acc_sq', 'flow_acc_pct', 'elev_x_flow']

Severn risk_0_2m distribution:
risk_0_2m
1.0     847433
2.0     761336
3.0     435579
4.0    1285362
Name: count, dtype: int64

Northumbria risk_0_2m distribution:
risk_0_2m
1.0    249498
2.0    395010
3.0    207453
4.0    564289
Name: count, dtype: int64

Class weights:
  Class 1: 0.982
  Class 2: 1.093
  Class 3: 1.911
  Class 4: 0.648

── v1: Severn validation (seen) ──
Accuracy: 0.479
              precision    recall  f1-score   support

    Very Low       0.42      0.55      0.48     10182
         Low       0.35      0.20     

In [5]:
# ── ERA5 Weather Feature Engineering ────────────────────────────────
# Goal: extract time-invariant precipitation extremes per spatial pixel
# Following RAPFLO (2024) and hackathon brief suggestions

import xarray as xr
import numpy as np
import pandas as pd

def engineer_weather_features(ds_era5):
    """
    Extract time-invariant extreme weather features from ERA5 daily data.
    Returns a dataframe with one row per spatial pixel.
    """
    # Convert to dataframe — shape: (time * y * x, variables)
    df_w = ds_era5[['tp', 'sro', 'swvl1_mean', 'swvl1_max']].to_dataframe().reset_index()
    df_w = df_w.dropna(subset=['tp'])
    
    print(f"Weather dataframe shape: {df_w.shape}")
    print(f"Unique spatial pixels: {df_w.groupby(['y','x']).ngroups}")
    
    # Group by spatial pixel and compute extremes over 10-year period
    grouped = df_w.groupby(['y', 'x'])
    
    weather_features = grouped.agg(
        # Precipitation extremes
        tp_p95       = ('tp', lambda x: np.percentile(x, 95)),
        tp_p99       = ('tp', lambda x: np.percentile(x, 99)),
        tp_mean      = ('tp', 'mean'),
        tp_max       = ('tp', 'max'),
        
        # Surface runoff extremes
        sro_p95      = ('sro', lambda x: np.percentile(x, 95)),
        sro_max      = ('sro', 'max'),
        sro_mean     = ('sro', 'mean'),
        
        # Soil moisture extremes
        swvl1_max_p95 = ('swvl1_max', lambda x: np.percentile(x, 95)),
        swvl1_min     = ('swvl1_mean', 'min'),  # dry periods
        swvl1_mean    = ('swvl1_mean', 'mean'),
    ).reset_index()
    
    # Rolling window features — max 5-day accumulated precipitation
    # Sort by pixel then time for rolling calculation
    df_w_sorted = df_w.sort_values(['y', 'x', 'valid_time'])
    df_w_sorted['tp_rolling5'] = (
        df_w_sorted.groupby(['y', 'x'])['tp']
        .transform(lambda x: x.rolling(5, min_periods=5).sum())
    )
    df_w_sorted['sro_rolling5'] = (
        df_w_sorted.groupby(['y', 'x'])['sro']
        .transform(lambda x: x.rolling(5, min_periods=5).sum())
    )
    
    rolling_features = df_w_sorted.groupby(['y', 'x']).agg(
        max_rolling5_tp  = ('tp_rolling5', 'max'),
        max_rolling5_sro = ('sro_rolling5', 'max'),
    ).reset_index()
    
    # Merge rolling into main weather features
    weather_features = weather_features.merge(rolling_features, on=['y', 'x'])
    
    print(f"Weather features shape: {weather_features.shape}")
    print(f"Columns: {weather_features.columns.tolist()}")
    
    return weather_features

print("── Engineering Severn weather features ──")
weather_severn = engineer_weather_features(ds_era5_severn)

print("\n── Engineering Northumbria weather features ──")
weather_northumbria = engineer_weather_features(ds_era5_northumbria)

print("\nSevern weather features sample:")
print(weather_severn.head())
print("\nStats:")
print(weather_severn.describe())

── Engineering Severn weather features ──
Weather dataframe shape: (774436, 8)
Unique spatial pixels: 212
Weather features shape: (212, 14)
Columns: ['y', 'x', 'tp_p95', 'tp_p99', 'tp_mean', 'tp_max', 'sro_p95', 'sro_max', 'sro_mean', 'swvl1_max_p95', 'swvl1_min', 'swvl1_mean', 'max_rolling5_tp', 'max_rolling5_sro']

── Engineering Northumbria weather features ──
Weather dataframe shape: (434707, 8)
Unique spatial pixels: 119
Weather features shape: (119, 14)
Columns: ['y', 'x', 'tp_p95', 'tp_p99', 'tp_mean', 'tp_max', 'sro_p95', 'sro_max', 'sro_mean', 'swvl1_max_p95', 'swvl1_min', 'swvl1_mean', 'max_rolling5_tp', 'max_rolling5_sro']

Severn weather features sample:
              y             x    tp_p95    tp_p99   tp_mean    tp_max  \
0  3.183421e+06  3.462569e+06  0.010958  0.018489  0.002434  0.038147   
1  3.183421e+06  3.472018e+06  0.010961  0.018242  0.002412  0.038413   
2  3.192870e+06  3.453120e+06  0.011224  0.018882  0.002496  0.036935   
3  3.192870e+06  3.462569e+06  0.

In [6]:
from scipy.spatial import cKDTree

def merge_weather_to_terrain(df_terrain, df_weather):
    """
    Merge coarse ERA5 weather features (9.45km grid) onto 
    fine terrain pixels (20m grid) using nearest neighbour.
    """
    # Build KD-tree from weather pixel coordinates
    weather_coords = df_weather[['y', 'x']].values
    terrain_coords = df_terrain[['y', 'x']].values
    
    tree = cKDTree(weather_coords)
    distances, indices = tree.query(terrain_coords, k=1)
    
    print(f"Max distance to nearest weather pixel: {distances.max():.0f}m")
    print(f"Mean distance to nearest weather pixel: {distances.mean():.0f}m")
    
    # Get weather features for each terrain pixel
    weather_cols = [c for c in df_weather.columns if c not in ['y', 'x']]
    weather_matched = df_weather.iloc[indices][weather_cols].reset_index(drop=True)
    
    # Concatenate
    df_merged = pd.concat([
        df_terrain.reset_index(drop=True),
        weather_matched
    ], axis=1)
    
    return df_merged

print("── Merging weather onto Severn terrain ──")
df_s_merged = merge_weather_to_terrain(df_s, weather_severn)

print("\n── Merging weather onto Northumbria terrain ──")
df_n_merged = merge_weather_to_terrain(df_n, weather_northumbria)

print(f"\nSevern merged shape: {df_s_merged.shape}")
print(f"Northumbria merged shape: {df_n_merged.shape}")

# ── Quick sanity check ────────────────────────────────────────────────
weather_feature_cols = [
    'tp_p95', 'tp_p99', 'tp_mean', 'tp_max',
    'sro_p95', 'sro_max', 'sro_mean',
    'swvl1_max_p95', 'swvl1_min', 'swvl1_mean',
    'max_rolling5_tp', 'max_rolling5_sro'
]

print("\nSevern weather feature nulls after merge:")
print(df_s_merged[weather_feature_cols].isnull().sum())

print("\nWeather feature correlations with risk_0_2m (Severn):")
corr = df_s_merged[weather_feature_cols + ['risk_0_2m']].corr()['risk_0_2m']
print(corr.sort_values())

# ── Model v3 — terrain + weather ─────────────────────────────────────
features_v3 = [
    # Terrain
    'dtm_zscore', 'log_flow_acc', 'log_flow_acc_sq',
    'flow_acc_pct', 'elev_x_flow',
    'imd', 'waw', 'is_waterway', 'clc_type_clean',
    # Weather
    'tp_p99', 'tp_max', 'max_rolling5_tp',
    'sro_p95', 'max_rolling5_sro',
    'swvl1_max_p95', 'swvl1_min'
]

df_sample3 = df_s_merged.dropna(subset=features_v3 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)

X3 = df_sample3[features_v3].values
y3 = (df_sample3['risk_0_2m'] - 1).astype(int).values

X_train3, X_val3, y_train3, y_val3 = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3
)

print(f"\nTraining v3 on {len(X_train3):,} samples...")

model3 = XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
model3.fit(X_train3, y_train3,
           eval_set=[(X_val3, y_val3)],
           verbose=False)

# Severn validation
y_pred3 = model3.predict(X_val3)
print("\n── v3: Severn validation (seen) ──")
print(f"Accuracy: {accuracy_score(y_val3, y_pred3):.3f}")
print(classification_report(y_val3, y_pred3,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

# Northumbria test
df_n3 = df_n_merged.dropna(subset=features_v3 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)
X_test3 = df_n3[features_v3].values
y_test3 = (df_n3['risk_0_2m'] - 1).astype(int).values

y_pred_n3 = model3.predict(X_test3)
print("── v3: Northumbria test (unseen) ──")
print(f"Accuracy: {accuracy_score(y_test3, y_pred_n3):.3f}")
print(classification_report(y_test3, y_pred_n3,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

# Feature importance
feat_imp3 = pd.Series(model3.feature_importances_, index=features_v3)
print("\n── Feature importance v3 ──")
print(feat_imp3.sort_values(ascending=False))

# ── Summary comparison ────────────────────────────────────────────────
print("\n── Accuracy progression ──")
print(f"v1 terrain only    — Severn: 0.479 | Northumbria: 0.417")
print(f"v2 expanded terrain — Severn: 0.493 | Northumbria: 0.418")
print(f"v3 + weather        — Severn: {accuracy_score(y_val3, y_pred3):.3f} | Northumbria: {accuracy_score(y_test3, y_pred_n3):.3f}")


── Merging weather onto Severn terrain ──
Max distance to nearest weather pixel: 10599m
Mean distance to nearest weather pixel: 3785m

── Merging weather onto Northumbria terrain ──
Max distance to nearest weather pixel: 13387m
Mean distance to nearest weather pixel: 4116m

Severn merged shape: (3329710, 35)
Northumbria merged shape: (1416255, 35)

Severn weather feature nulls after merge:
tp_p95              0
tp_p99              0
tp_mean             0
tp_max              0
sro_p95             0
sro_max             0
sro_mean            0
swvl1_max_p95       0
swvl1_min           0
swvl1_mean          0
max_rolling5_tp     0
max_rolling5_sro    0
dtype: int64

Weather feature correlations with risk_0_2m (Severn):
tp_mean            -0.145385
tp_p95             -0.082311
sro_p95            -0.075270
tp_max             -0.054519
sro_mean           -0.049901
sro_max            -0.022949
swvl1_mean          0.000176
swvl1_min           0.000594
swvl1_max_p95       0.016263
max_rolling5_s

In [8]:
# ── Fix: Region-relative weather normalization ────────────────────────
# Instead of absolute weather values, use z-scores within each region
# This makes the features transferable across regions

def add_relative_weather(df_merged, weather_cols):
    df = df_merged.copy()
    for col in weather_cols:
        mean = df[col].mean()
        std  = df[col].std()
        df[f'{col}_zscore'] = (df[col] - mean) / (std + 1e-8)
    return df

weather_feature_cols = [
    'tp_p95', 'tp_p99', 'tp_mean', 'tp_max',
    'sro_p95', 'sro_max', 'sro_mean',
    'swvl1_max_p95', 'swvl1_min', 'swvl1_mean',
    'max_rolling5_tp', 'max_rolling5_sro'
]

df_s_merged = add_relative_weather(df_s_merged, weather_feature_cols)
df_n_merged = add_relative_weather(df_n_merged, weather_feature_cols)

# RAPFLO-inspired: multiply relative weather × terrain features
# High precipitation matters more in low-elevation, high-flow areas
for df in [df_s_merged, df_n_merged]:
    df['tp_x_flow']       = df['tp_p99_zscore'] * df['log_flow_acc']
    df['rolling_x_flow']  = df['max_rolling5_tp_zscore'] * df['log_flow_acc']
    df['sro_x_elev']      = df['sro_p95_zscore'] * df['dtm_zscore']
    df['swvl_x_flow']     = df['swvl1_max_p95_zscore'] * df['log_flow_acc']

# ── Model v4 — relative weather + interactions ────────────────────────
features_v4 = [
    # Terrain
    'dtm_zscore', 'log_flow_acc', 'elev_x_flow',
    'imd', 'waw', 'is_waterway', 'clc_type_clean',
    # Relative weather z-scores
    'tp_p99_zscore', 'tp_max_zscore', 'max_rolling5_tp_zscore',
    'sro_p95_zscore', 'max_rolling5_sro_zscore',
    'swvl1_max_p95_zscore', 'swvl1_min_zscore',
    # RAPFLO-style interactions
    'tp_x_flow', 'rolling_x_flow', 'sro_x_elev', 'swvl_x_flow'
]

df_sample4 = df_s_merged.dropna(subset=features_v4 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)

X4 = df_sample4[features_v4].values
y4 = (df_sample4['risk_0_2m'] - 1).astype(int).values

X_train4, X_val4, y_train4, y_val4 = train_test_split(
    X4, y4, test_size=0.2, random_state=42, stratify=y4
)

model4 = XGBClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
model4.fit(X_train4, y_train4,
           eval_set=[(X_val4, y_val4)],
           verbose=False)

y_pred4 = model4.predict(X_val4)
print("── v4: Severn validation (seen) ──")
print(f"Accuracy: {accuracy_score(y_val4, y_pred4):.3f}")
print(classification_report(y_val4, y_pred4,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

df_n4 = df_n_merged.dropna(subset=features_v4 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)
X_test4 = df_n4[features_v4].values
y_test4 = (df_n4['risk_0_2m'] - 1).astype(int).values

y_pred_n4 = model4.predict(X_test4)
print("── v4: Northumbria test (unseen) ──")
print(f"Accuracy: {accuracy_score(y_test4, y_pred_n4):.3f}")
print(classification_report(y_test4, y_pred_n4,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

feat_imp4 = pd.Series(model4.feature_importances_, index=features_v4)
print("\n── Feature importance v4 ──")
print(feat_imp4.sort_values(ascending=False))

print("\n── Accuracy progression ──")
print(f"v1 terrain only       — Severn: 0.479 | Northumbria: 0.417")
print(f"v2 expanded terrain   — Severn: 0.493 | Northumbria: 0.418")
print(f"v3 absolute weather   — Severn: 0.586 | Northumbria: 0.421")
print(f"v4 relative weather   — Severn: {accuracy_score(y_val4, y_pred4):.3f} | Northumbria: {accuracy_score(y_test4, y_pred_n4):.3f}")

── v4: Severn validation (seen) ──
Accuracy: 0.582
              precision    recall  f1-score   support

    Very Low       0.52      0.73      0.61     10182
         Low       0.48      0.41      0.44      9125
      Medium       0.46      0.11      0.18      5261
        High       0.69      0.75      0.72     15432

    accuracy                           0.58     40000
   macro avg       0.54      0.50      0.49     40000
weighted avg       0.57      0.58      0.56     40000

── v4: Northumbria test (unseen) ──
Accuracy: 0.427
              precision    recall  f1-score   support

    Very Low       0.24      0.09      0.13     35347
         Low       0.36      0.29      0.32     55778
      Medium       0.26      0.07      0.11     29449
        High       0.48      0.80      0.60     79426

    accuracy                           0.43    200000
   macro avg       0.34      0.31      0.29    200000
weighted avg       0.37      0.43      0.37    200000


── Feature importance v4 ─

In [ ]:
# ── Spatial block sampling — reduces spatial autocorrelation ──────────
# Divide Severn into spatial blocks, sample uniformly across blocks
# This forces the model to generalize across space within Severn
# which should improve Northumbria generalization

def spatial_block_sample(df, n_samples, block_size=10000, random_state=42):
    """
    Sample pixels such that each spatial block contributes equally.
    block_size is in coordinate units (metres in EPSG:3035).
    """
    df = df.copy()
    df['block_y'] = (df['y'] // block_size).astype(int)
    df['block_x'] = (df['x'] // block_size).astype(int)
    df['block_id'] = df['block_y'].astype(str) + '_' + df['block_x'].astype(str)
    
    n_blocks = df['block_id'].nunique()
    per_block = max(1, n_samples // n_blocks)
    
    sampled = (df.groupby('block_id', group_keys=False)
                 .apply(lambda x: x.sample(min(len(x), per_block), 
                                           random_state=random_state)))
    
    # If under-sampled, top up randomly
    if len(sampled) < n_samples:
        remaining = df[~df.index.isin(sampled.index)]
        extra = remaining.sample(
            min(n_samples - len(sampled), len(remaining)), 
            random_state=random_state
        )
        sampled = pd.concat([sampled, extra])
    
    print(f"  Blocks: {n_blocks} | Per block: {per_block} | Total sampled: {len(sampled):,}")
    return sampled.drop(columns=['block_y', 'block_x', 'block_id'])

print("Spatial block sampling for Severn training set:")
df_sample5 = spatial_block_sample(
    df_s_merged.dropna(subset=features_v4 + ['risk_0_2m']), 
    n_samples=200_000
)

X5 = df_sample5[features_v4].values
y5 = (df_sample5['risk_0_2m'] - 1).astype(int).values

X_train5, X_val5, y_train5, y_val5 = train_test_split(
    X5, y5, test_size=0.2, random_state=42, stratify=y5
)

model5 = XGBClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
model5.fit(X_train5, y_train5,
           eval_set=[(X_val5, y_val5)],
           verbose=False)

y_pred5 = model5.predict(X_val5)
print("\n── v5: Severn validation (seen) ──")
print(f"Accuracy: {accuracy_score(y_val5, y_pred5):.3f}")
print(classification_report(y_val5, y_pred5,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

df_n5 = df_n_merged.dropna(subset=features_v4 + ['risk_0_2m']).sample(
    n=200_000, random_state=42
)
X_test5 = df_n5[features_v4].values
y_test5 = (df_n5['risk_0_2m'] - 1).astype(int).values

y_pred_n5 = model5.predict(X_test5)
print("── v5: Northumbria test (unseen) ──")
print(f"Accuracy: {accuracy_score(y_test5, y_pred_n5):.3f}")
print(classification_report(y_test5, y_pred_n5,
      target_names=['Very Low', 'Low', 'Medium', 'High']))

print("\n── Accuracy progression ──")
print(f"v1 terrain only         — Severn: 0.479 | Northumbria: 0.417")
print(f"v2 expanded terrain     — Severn: 0.493 | Northumbria: 0.418")
print(f"v3 absolute weather     — Severn: 0.586 | Northumbria: 0.421")
print(f"v4 relative weather     — Severn: 0.582 | Northumbria: 0.427")
print(f"v5 spatial block sample — Severn: {accuracy_score(y_val5, y_pred5):.3f} | Northumbria: {accuracy_score(y_test5, y_pred_n5):.3f}")